<h1 style='text-align:center; color:#e94560;'>🎬 Movie Mind</h1>
<p style='text-align:center; color:#8899aa; font-size:1.1rem;'>Multi-modal vector movie recommender — Semantic · TF-IDF · Hybrid Ensemble</p>

In [ ]:
# Fix protobuf C extension issue BEFORE any imports
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

import json, pickle, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
warnings.filterwarnings('ignore')

BASE = Path.cwd()
DATA = BASE / 'data'
INDEX_DIR = BASE / 'index'
print('✅ imports ready')

## ⚙️ Build Vector Index
Run this cell once to create the ChromaDB index from the TMDB dataset.

In [ ]:
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# ── Load & parse ──
def json_parse(val):
    if pd.isna(val) or val == '': return []
    try: return json.loads(val) if isinstance(val, str) else val
    except: return []

movies = pd.read_csv(DATA / 'tmdb_5000_movies.csv')
credits = pd.read_csv(DATA / 'tmdb_5000_credits.csv')

for col in ['genres','keywords','production_companies','production_countries','spoken_languages']:
    movies[col] = movies[col].apply(json_parse)
movies['genre_names'] = movies['genres'].apply(lambda g: [x['name'] for x in g] if isinstance(g,list) else [])
movies['keyword_names'] = movies['keywords'].apply(lambda k: [x['name'] for x in k] if isinstance(k,list) else [])

credits['cast_json'] = credits['cast'].apply(lambda x: json_parse(x) if isinstance(x,str) else [])
credits['crew_json'] = credits['crew'].apply(lambda x: json_parse(x) if isinstance(x,str) else [])
credits['top_cast'] = credits['cast_json'].apply(lambda c: [x['name'] for x in c[:5]] if isinstance(c,list) else [])
credits['director'] = credits['crew_json'].apply(
    lambda c: next((x['name'] for x in c if isinstance(x,dict) and x.get('job')=='Director'), '') if isinstance(c,list) else '')

df = movies.merge(credits[['movie_id','top_cast','director']], left_on='id', right_on='movie_id', how='left')
df['top_cast'] = df['top_cast'].apply(lambda c: c if isinstance(c,list) else [])
df['director'] = df['director'].fillna('')

# ── Numerical features ──
for c in ['budget','revenue','runtime','popularity']:
    df[c] = df[c].fillna(0).astype(float)
df['vote_count'] = df['vote_count'].fillna(0).astype(int)
df['vote_average'] = df['vote_average'].fillna(0).astype(float)
df['profit'] = df['revenue'] - df['budget']
df['roi'] = df.apply(lambda r: r['profit']/r['budget'] if r['budget']>0 else 0, axis=1)
df['popularity_log'] = np.log1p(df['popularity'])
df['budget_log'] = np.log1p(df['budget'])
df['revenue_log'] = np.log1p(df['revenue'])

# ── Year / recency ──
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['year'] = df['release_date'].dt.year.fillna(0).astype(int)
df['age_days'] = (pd.Timestamp.now() - df['release_date']).dt.days.fillna(0)
df['recency_score'] = 1 / (1 + df['age_days'] / 365.25)

# ── Bayesian weighted rating ──
C = df['vote_average'].mean()
m = df['vote_count'].quantile(0.90)
df['vote_weighted'] = (df['vote_count']*df['vote_average'] + m*C) / (df['vote_count'] + m)
df['is_english'] = (df['original_language']=='en').astype(int)

# Filter to movies with overviews
df = df[df['overview'].notna() & (df['overview']!='')].copy()
print(f'✅ {len(df)} movies loaded with overviews')

In [ ]:
# ── Build text for embedding ──
def build_text(movie):
    parts = []
    for f in ['title','tagline','overview']:
        v = movie.get(f,'')
        if v and isinstance(v,str) and v.strip():
            parts.append(f'{f.capitalize()}: {v.strip()}')
    for label,col in [('Genres','genre_names'),('Keywords','keyword_names'),('Cast','top_cast')]:
        vals = movie.get(col,[])
        if vals and isinstance(vals,list):
            parts.append(f'{label}: {", ".join(str(v) for v in vals[:8])}')
    if movie.get('director',''):
        parts.append(f'Director: {movie["director"]}')
    return ' | '.join(parts)

texts = df.apply(build_text, axis=1).tolist()
all_genres = sorted(set(g for gs in df['genre_names'] for g in gs))

# ── Layer 1: Sentence embeddings ──
print('Generating sentence embeddings...')
text_model = SentenceTransformer('all-MiniLM-L6-v2')
sent_emb = text_model.encode(texts, show_progress_bar=True, batch_size=64)
print(f'  Semantic: {sent_emb.shape}')

# ── Layer 2: TF-IDF ──
print('Building TF-IDF...')
tfidf = TfidfVectorizer(max_features=2000, stop_words='english', ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['overview'].fillna(''))
svd = TruncatedSVD(n_components=50, random_state=42)
tfidf_latent = svd.fit_transform(tfidf_matrix)
print(f'  TF-IDF latent: {tfidf_latent.shape}')

# ── Layer 3: Hybrid vector ──
def hybrid_vec(movie):
    v = [movie.get('popularity_log',0), movie.get('budget_log',0), movie.get('revenue_log',0),
         movie.get('vote_weighted',0), movie.get('recency_score',0), movie.get('runtime',0)/300,
         movie.get('is_english',0), movie.get('roi',0)/100]
    v.extend([1 if g in movie.get('genre_names',[]) else 0 for g in all_genres])
    return v

hybrid_vecs = np.array([hybrid_vec(row) for _,row in df.iterrows()])
scaler = StandardScaler()
hybrid_scaled = scaler.fit_transform(hybrid_vecs)
print(f'  Hybrid features: {hybrid_scaled.shape}')

# ── Ensemble ──
sent_n = sent_emb / (np.linalg.norm(sent_emb, axis=1, keepdims=True)+1e-8)
tfidf_n = tfidf_latent / (np.linalg.norm(tfidf_latent, axis=1, keepdims=True)+1e-8)
hybrid_n = hybrid_scaled / (np.linalg.norm(hybrid_scaled, axis=1, keepdims=True)+1e-8)
ensemble = np.concatenate([sent_n*0.5, tfidf_n*0.2, hybrid_n*0.3], axis=1)
print(f'  Ensemble: {ensemble.shape}')

# ── Build ChromaDB ──
client = chromadb.PersistentClient(path=str(INDEX_DIR), settings=Settings(anonymized_telemetry=False))
for name in ['movies_semantic','movies_ensemble']:
    try: client.delete_collection(name)
    except: pass

col_sem = client.create_collection('movies_semantic', metadata={'hnsw:space':'cosine'})
col_ens = client.create_collection('movies_ensemble', metadata={'hnsw:space':'cosine'})
ids = [str(x) for x in df['id'].tolist()]
meta_list = [{'title':r['title'],'id':int(r['id']),'year':int(r['year']) if r['year'] else 0,
              'rating':float(r['vote_average']), 'genres':','.join(r['genre_names'][:4])}
             for _,r in df.iterrows()]

for i in range(0, len(ids), 100):
    end = min(i+100, len(ids))
    col_sem.add(embeddings=sent_emb[i:end].tolist(), ids=ids[i:end], metadatas=meta_list[i:end], documents=texts[i:end])
    col_ens.add(embeddings=ensemble[i:end].tolist(), ids=ids[i:end], metadatas=meta_list[i:end])
print(f'✅ Index built: {col_sem.count()} movies (semantic), {col_ens.count()} (ensemble)')

In [ ]:
# ── Save artifacts for future sessions ──
INDEX_DIR.mkdir(parents=True, exist_ok=True)
artifacts = {
    'all_genres': all_genres,
    'ids': ids, 'titles': df['title'].tolist(),
    'meta': df[['id','title','vote_average','vote_count','vote_weighted','year','popularity','runtime','revenue','budget','genre_names','keyword_names','top_cast','director','overview','tagline','recency_score']].to_dict('records'),
}
with open(INDEX_DIR/'artifacts.pkl','wb') as f: pickle.dump(artifacts, f)
text_model.save(str(INDEX_DIR/'sentence_model'))
with open(INDEX_DIR/'tfidf.pkl','wb') as f: pickle.dump(tfidf, f)
with open(INDEX_DIR/'svd.pkl','wb') as f: pickle.dump(svd, f)
with open(INDEX_DIR/'scaler.pkl','wb') as f: pickle.dump(scaler, f)
print('✅ All artifacts saved')

---
## 🔍 Semantic Search
Describe what you want to watch and find matching movies.

In [ ]:
def recommend_text(query, n=10):
    emb = text_model.encode([query])[0].tolist()
    raw = col_sem.query(query_embeddings=[emb], n_results=n*3, include=['distances','metadatas'])
    scored = []
    for i,(mid,dist,m) in enumerate(zip(raw['ids'][0], raw['distances'][0], raw['metadatas'][0])):
        mm = artifacts['meta'][artifacts['ids'].index(mid)] if mid in artifacts['ids'] else None
        if not mm: continue
        sim = 1 - dist
        hybrid = 0.45*sim + 0.20*((mm.get('vote_weighted',5)-5)/5) + 0.20*min(mm.get('popularity',0)/20,1) + 0.15*mm.get('recency_score',0.5)
        scored.append((mm, sim, hybrid))
    scored.sort(key=lambda x: x[2], reverse=True)
    return scored[:n]

def show_movies(results):
    data = []
    for mm, sim, hybrid in results:
        data.append({
            'Title': mm['title'],
            'Score': f'{hybrid:.0%}',
            'Rating': f'{mm["vote_average"]:.1f}' if mm['vote_average'] else '-',
            'Year': str(mm['year']) if mm.get('year') else '-',
            'Genres': ', '.join(mm.get('genre_names',[])[:3]),
            'Director': mm.get('director',''),
        })
    return pd.DataFrame(data)

# ── ENTER YOUR QUERY HERE ──
QUERY = 'A mind-bending sci-fi movie with philosophical themes'

results = recommend_text(QUERY)
print(f'Top picks for: "{QUERY}"\n')
show_movies(results)

---
## 🎯 Similar Movies
Pick a movie you like and find similar ones.

In [ ]:
# ── ENTER A MOVIE TITLE ──
MOVIE_TITLE = 'Inception'

# Find the movie
matches = [m for m in artifacts['meta'] if MOVIE_TITLE.lower() in m['title'].lower()]
if not matches:
    print(f'Movie "{MOVIE_TITLE}" not found. Try another title.')
else:
    chosen = matches[0]
    mid = chosen['id']
    
    ens_result = col_ens.get(ids=[str(mid)], include=['embeddings'])
    if ens_result['embeddings']:
        raw = col_ens.query(query_embeddings=[ens_result['embeddings'][0]], n_results=15, include=['distances','metadatas'])
        scored = []
        for mid2,dist,m in zip(raw['ids'][0], raw['distances'][0], raw['metadatas'][0]):
            if int(mid2) == mid: continue
            cand = artifacts['meta'][artifacts['ids'].index(mid2)]
            sim = 1 - dist
            qg = set(chosen.get('genre_names',[]))
            cg = set(cand.get('genre_names',[]))
            overlap = len(qg & cg) / max(len(qg | cg), 1)
            hybrid = 0.50*sim + 0.15*overlap + 0.20*(cand.get('vote_weighted',5)/10) + 0.15*min(cand.get('popularity',0)/20,1)
            scored.append((cand, sim, hybrid))
        scored.sort(key=lambda x: x[2], reverse=True)
        print(f'Because you liked: {chosen["title"]}\n')
        show_movies(scored[:10])

---
## 🎭 Multi-Movie Blend
Pick 2-3 movies and find recommendations that blend their styles.

In [ ]:
# ── ENTER 2-3 MOVIE TITLES ──
MOVIES = ['The Matrix', 'Inception', 'Interstellar']

embeddings = []
found_titles = []
for t in MOVIES:
    match = next((m for m in artifacts['meta'] if t.lower() in m['title'].lower()), None)
    if match:
        r = col_ens.get(ids=[str(match['id'])], include=['embeddings'])
        if r['embeddings']:
            embeddings.append(r['embeddings'][0])
            found_titles.append(match['title'])

if len(embeddings) >= 1:
    centroid = np.mean(embeddings, axis=0).tolist()
    raw = col_ens.query(query_embeddings=[centroid], n_results=15, include=['distances','metadatas'])
    scored = []
    for mid,dist,m in zip(raw['ids'][0], raw['distances'][0], raw['metadatas'][0]):
        if int(mid) in [artifacts['meta'][artifacts['ids'].index(str(fid))]['id'] for fid in [m['id'] for m in [next(x for x in artifacts['meta'] if t.lower() in x['title'].lower()) for t in MOVIES if next((x for x in artifacts['meta'] if t.lower() in x['title'].lower()), None)]]]:
            continue
        cand = artifacts['meta'][artifacts['ids'].index(mid)]
        scored.append((cand, 1-dist, 1-dist))
    scored.sort(key=lambda x: x[2], reverse=True)
    print(f'Blending: {", ".join(found_titles)}\n')
    show_movies(scored[:10])
else:
    print('Could not find any of those movies.')

---
## ⭐ Curated Picks
Highest-rated movies with genre diversity.

In [ ]:
meta_df = pd.DataFrame(artifacts['meta'])
meta_df['score'] = (meta_df['vote_weighted'] * 0.4 + np.log1p(meta_df['popularity']) / 5 * 0.3 + meta_df['recency_score'] * 0.3)
top = meta_df.nlargest(30, 'score')

seen_genres = set()
picks = []
for _, r in top.iterrows():
    gs = frozenset(r.get('genre_names', []))
    if gs not in seen_genres or len(picks) < 10:
        seen_genres.add(gs)
        picks.append(r)

result_df = pd.DataFrame([{
    'Title': r['title'],
    'Rating': f'{r["vote_average"]:.1f}' if r['vote_average'] else '-',
    'Weighted': f'{r["vote_weighted"]:.2f}',
    'Year': str(r['year']) if r.get('year') else '-',
    'Genres': ', '.join(r.get('genre_names', [])[:3]),
} for _, r in top.iterrows()])
print('🏆 Top picks\n')
result_df.head(12)

---
## 🔬 Explore the Data


In [ ]:
# Top genres
from collections import Counter
genre_counts = Counter(g for gs in df['genre_names'] for g in gs)
genre_df = pd.DataFrame(genre_counts.most_common(15), columns=['Genre', 'Count'])
genre_df.style.bar(subset=['Count'], color='#e94560')

In [ ]:
# Rating distribution
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['vote_average'], bins=20, color='#e94560', edgecolor='white', alpha=0.8)
axes[0].set_title('Rating Distribution', color='white')
axes[0].set_xlabel('Rating', color='#8899aa')
axes[0].set_ylabel('Count', color='#8899aa')
axes[0].tick_params(colors='#8899aa')
axes[0].set_facecolor('#1a1a2e')

axes[1].hist(df['year'], bins=30, color='#0f3460', edgecolor='white', alpha=0.8)
axes[1].set_title('Movies by Year', color='white')
axes[1].set_xlabel('Year', color='#8899aa')
axes[1].tick_params(colors='#8899aa')
axes[1].set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#0a0a1a')
plt.tight_layout()
plt.show()

---
## 📁 Data & Model Info

In [ ]:
print(f'''{'='*50}
📊 MOVIE MIND — DATASET SUMMARY
{'='*50}
Total movies indexed: {len(df)}
Total genres: {len(all_genres)}
Date range: {int(df['year'].min())} – {int(df['year'].max())}
Avg rating: {df['vote_average'].mean():.2f}
Embedding dimensions:
  Semantic (MiniLM): {sent_emb.shape[1]}
  TF-IDF latent: {tfidf_latent.shape[1]}
  Hybrid features: {hybrid_scaled.shape[1]}
  Ensemble total: {ensemble.shape[1]}
{'='*50}
''')

print('📦 To reload this notebook later, run the first cell then:')
print('  client = chromadb.PersistentClient(path="index", settings=Settings(anonymized_telemetry=False))')
print('  col_sem = client.get_collection("movies_semantic")')
print('  col_ens = client.get_collection("movies_ensemble")')
print('  text_model = SentenceTransformer("index/sentence_model")')
print('  with open("index/artifacts.pkl","rb") as f: artifacts = pickle.load(f)')